In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from mdsetup import MDSetup

# Change to the correct directory
os.chdir('')

# Initialize MDSetup
lammps_setup = MDSetup(
    system_setup="input_tob_all/setup_mechanical_pcff.yaml",
    simulation_default="input_tob_all/defaults.yaml",
    simulation_ensemble="input_tob_all/ensemble.yaml",
    simulation_sampling="input_tob_all/sampling_mechanical.yaml",
    submission_command="qsub",
)


# Define paths
LJ_SETS = ['IFF_reproduce/PCFF']
TOB_STRUCTURES = ["Tob9", "Tob11", "Tob11H", "Tob14"]


In [ ]:
# Define ensemble and analysis parameters
ENSEMBLE = "01_npt"
TIME_FRACTION = 0.2  # Percentage to discard from the beginning of the simulation
PROPERTIES = {
    # "lattice": ["a", "b", "c", "alpha", "beta", "gamma"],
    "density": ["density"],
    # "energy": ["potential energy"],
}

# Storage dictionary for results
results = {prop: {tob: [] for tob in TOB_STRUCTURES} for prop in PROPERTIES}

# Loop over LJ parameter sets and Tob structures
for lj_set in LJ_SETS:
    for tob_structure in TOB_STRUCTURES:
        analysis_folder = f"{tob_structure}/{lj_set}/equilibration"
        
        # Store results for the current LJ set
        lj_results = {prop: [] for prop in PROPERTIES}

        for output_suffix, props in PROPERTIES.items():
            extracted_values = lammps_setup.analysis_extract_properties(
                analysis_folder=analysis_folder,
                ensemble=ENSEMBLE,
                extracted_properties=props,
                output_suffix=output_suffix,
                time_fraction=TIME_FRACTION,
            )
            average_values = extracted_values.get(ENSEMBLE, {}).get("data", {}).get("average", {})

            # Extract required values
            for prop in props:
                mean_value = average_values.get(prop, {}).get("mean", None)
                lj_results[output_suffix].append(mean_value)

        # Append results for this LJ set
        for prop in PROPERTIES:
            results[prop][tob_structure].append(lj_results[prop])



In [ ]:
results

In [ ]:
# Define parameters for deformation analysis
ENSEMBLE = "00_nvt"
DEFORMATION_RATES = [-0.02, -0.01, 0.00, 0.01, 0.02]
TIME_FRACTION = 0.4
METHOD = "VRH"
VISUALIZE_STRESS_STRAIN = False  # Change to True if you want plots per simulation

# Dictionary to hold Bulk Modulus values
bm_results = {tob: [] for tob in TOB_STRUCTURES}

# Loop through each LJ set and Tob structure
for lj_set in LJ_SETS:
    for tob_structure in TOB_STRUCTURES:
        # Define the deformation analysis folder
        analysis_folder = f"{tob_structure}/{lj_set}/deformation"

        # Run analysis
        BM = lammps_setup.analysis_mechanical_proerties(
            analysis_folder=analysis_folder,
            ensemble=ENSEMBLE,
            deformation_rates=DEFORMATION_RATES,
            method=METHOD,
            time_fraction=TIME_FRACTION,
            visualize_stress_strain=VISUALIZE_STRESS_STRAIN,
        )

        print(f"✅ Analyzed {tob_structure} with {lj_set}: BM = {BM:.2f} GPa")
        bm_results[tob_structure].append(BM)



In [ ]:
bm_results

In [ ]:
# Constants
NA = 6.022e23
CONVERSION = 4184  # kcal/mol to J/mol

# Define structures and LJ param sets


# Box dimensions (structure-dependent)
box_coords = {
    "Tob9":     [-0.378508016, 21.933491984, -0.012474231, 21.896525769],
    "Tob11":    [0.527972175, 23.057572175, -0.431033519, 21.723966481],
    "Tob11H":   [0.065633394, 22.383633394, -0.317852898, 29.242147102],
    "Tob14":    [-0.658063909, 21.871536091, -0.289451837, 21.985548163],
}

# SE results
se_results = {struct: [] for struct in TOB_STRUCTURES}

# Loop over all structures and LJ param sets
for struct in TOB_STRUCTURES:
    for lj in LJ_SETS:
        base = f"{struct}/{lj}/SE"
        
        state = 'bulk'
        folder = f"{base}/{state}"
        extracted_values = lammps_setup.analysis_extract_properties(
            analysis_folder=folder,
            ensemble="00_nvt",
            extracted_properties=["potential energy"],
            output_suffix="energy",
            time_fraction=0.4,
        )
        bulk_energy = extracted_values.get("00_nvt", {}).get("data", {}).get("average", {}).get("potential energy", {})

        state = 'vacuum'
        folder = f"{base}/{state}"
        extracted_values = lammps_setup.analysis_extract_properties(
            analysis_folder=folder,
            ensemble="00_nvt",
            extracted_properties=["potential energy"],
            output_suffix="energy",
            time_fraction=0.4,
        )
        vac_energy = extracted_values.get("00_nvt", {}).get("data", {}).get("average", {}).get("potential energy", {})
        
        E_bulk, SD_bulk = bulk_energy["mean"], bulk_energy["std"]
        E_vac, SD_vac = vac_energy["mean"], vac_energy["std"]

        # Compute surface area
        xlo, xhi, ylo, yhi = box_coords[struct]
        A = abs(xhi - xlo) * abs(yhi - ylo) * 1e-20  # m²

        # Compute SE and std dev
        deltaE = 1000 * (E_vac - E_bulk) * CONVERSION / (2 * A * NA)  # mJ/m²
        s_dev = 1000 * (SD_bulk + SD_vac) * CONVERSION / (2 * A * NA)

        se_results[struct].append((deltaE, s_dev))


In [ ]:
se_results

In [ ]:
se_results